# Recursive world models — Experiment 4 · adaptive knowledge revision
25 November 2026. CPU only. No GPU, no network, no weights. Continuation of research/recursive-world-model.
## Historical ledger
- E1 ls20 24 observations: frozen WARC 78.56% changed-cell accuracy; direct adapter 86.54%; recursive dictionary transfer 79.37%; scratch dictionary 32.29%. No full WARC TTT comparison. Soft slots duplicated primitives; growth did not establish semantics. Hard execution on synthetic data failed. Do not tune this failed executor.
- E2 retained neural baseline, introduced abstaining executable corrections; CPU correctness passed; real-data benefit still unestablished.
- E3 finite 4-cell 3-color domain: learned macro prior 98.9% vs factorized 85.7% one-example unseen-grid accuracy on replication. Shifted family 54.3% vs 78.9%. This is ranking a fixed candidate bank, not unrestricted invention.
## E4 hypothesis and frozen forecasts
Reuse previously learned *joint rewrite distributions* as latent contexts, rather than destroy them during changes. Compare uniform, fixed factorized/joint, forgetting, Bayesian change-point filtering, context factorized/joint. All receive same 48 meta-training tasks (except uniform) and same one-example test task observations. After scoring, all receive full target truth table to infer compatible rewrites; this is a generous synthetic feedback protocol, not ordinary ARC. Stable / A-B-A / random streams, 60 tasks each, 48 streams per seed. No context labels given to learner. 
Forecasts before execution: stable >=96% exact on unseen inputs, first shift recovery <=10 tasks, return <=2 tasks, no meaningful advantage on random tasks. Recovery = first offset of three consecutive across-stream mean accuracies >=90%. Report mean paired differences and stream bootstrap. No tuning on evaluation seeds.
Architecture: observed source + one target pair -> enumerate candidate transformations -> prior from latent library -> consistent candidate -> predict unseen inputs -> full task evidence -> update posterior or allocate context. Existing contexts retained. Change-point baseline instead tracks distribution over run lengths; no stored regime library. Both use fractional-credit counts for ambiguous programs, not exact Dirichlet inference. Context allocation is hard MAP and fixed hazard/stickiness, not learned revision procedures.

In [ ]:
import itertools
import numpy as np
import pandas as pd
from time import perf_counter
DOMAIN=np.array(list(itertools.product(range(3),repeat=4)),dtype=np.int8)
N=len(DOMAIN)
POWERS=np.array([27,9,3,1])
def index(x): return x@POWERS
PRIMITIVES={
 'left':np.roll(DOMAIN,-1,axis=1),'right':np.roll(DOMAIN,1,axis=1),
 'reverse':DOMAIN[:,::-1].copy(),'cycle':(DOMAIN+1)%3,'uncycle':(DOMAIN+2)%3,
 'swap01':np.array([1,0,2],dtype=np.int8)[DOMAIN],
 'swap12':np.array([0,2,1],dtype=np.int8)[DOMAIN],
 'swap_halves':DOMAIN[:,[2,3,0,1]]}
TRANSFORMS=[('before',p) for p in PRIMITIVES]+[('after',p) for p in PRIMITIVES]+[('guard',c) for c in range(3)]+[('identity',None)]
def transform(program,spec):
    kind,arg=spec
    if kind=='before': return program[index(PRIMITIVES[arg])]
    if kind=='after': return PRIMITIVES[arg][index(program)]
    if kind=='guard': return np.where(DOMAIN==arg,program,DOMAIN)
    if kind=='identity': return program.copy()
    raise ValueError(spec)
def candidates(source): return np.stack([transform(source,t) for t in TRANSFORMS])
def consistent(bank,ids,outputs): return np.all(bank[:,ids]==outputs[None],axis=(1,2))
def acquire_source(observed):
    bank=np.stack(list(PRIMITIVES.values()))
    alive=consistent(bank,np.arange(N),observed)
    assert alive.any()
    return bank[alive][0].copy()
TRAIN_BASES=['left','cycle','swap01','reverse']
TEST_BASES=['right','uncycle','swap12','swap_halves']
MACROS=[(i,j) for i in range(len(TRANSFORMS)) for j in range(len(TRANSFORMS))]
def macro_bank(source):
    return np.concatenate([candidates(source),np.stack([transform(transform(source,TRANSFORMS[i]),TRANSFORMS[j]) for i,j in MACROS])])
PAIR_FAMILIES={
 'matched':[(('guard',0),('after','cycle')),(('guard',1),('after','swap01')),(('guard',2),('after','reverse'))],
 'swapped':[(('guard',0),('after','swap01')),(('guard',1),('after','reverse')),(('guard',2),('after','cycle'))]}
def apply_pair(s,pair): return transform(transform(s,pair[0]),pair[1])
def learn_macros(family,seed=17,n=48):
    rng=np.random.default_rng(seed); m=len(TRANSFORMS); c=np.ones(m+m*m)
    for ep in range(n):
        s=acquire_source(PRIMITIVES[TRAIN_BASES[ep%len(TRAIN_BASES)]])
        pair=family[int(rng.integers(len(family)))]; y=apply_pair(s,pair)
        alive=consistent(macro_bank(s),np.arange(N),y)
        assert alive.any(); c+=alive/alive.sum()
    joint=c[m:].reshape(m,m); f=np.outer(joint.sum(1),joint.sum(0))/joint.sum()
    factor=np.concatenate([c[:m],f.ravel()])
    assert np.isclose(c.sum(),factor.sum())
    return c,factor


In [ ]:
# Same candidate bank as revision 3. All arms get exactly the same observations.
M=len(TRANSFORMS); D=M+M*M; U=np.ones(D)/D
HAZARD=.05; DECAY=.9; STICKY=.90

def norm(v): return v/v.sum(axis=-1,keepdims=True)
def factor(v):
    j=v[M:].reshape(M,M)
    return np.r_[v[:M],(np.outer(j.sum(1),j.sum(0))/j.sum()).ravel()]

class Fixed:
    def __init__(self,c): self.c=c.copy()
    def prior(self): return norm(self.c+U)
    def update(self,e): pass

class Forget(Fixed):
    def update(self,e): self.c=DECAY*self.c+e/e.sum()

class ChangePoint:
    # Full run-length support, no pruning; fractional-credit count update.
    def __init__(self,c): self.c=c[None].copy(); self.w=np.ones(1)
    def prior(self): return (1-HAZARD)*(self.w@norm(self.c+U))+HAZARD*U
    def update(self,e):
        p=norm(self.c+U); likelihood=p@e
        self.w=norm(np.r_[HAZARD*(U@e),(1-HAZARD)*self.w*likelihood])
        credit=e/e.sum(); self.c=np.vstack([credit,self.c+credit])

class Context:
    # A new context competes with previously learned contexts. No context IDs.
    def __init__(self,c,factorized=False):
        self.c=[c.copy()]; self.last=0; self.factorized=factorized; self.births=0; self.switches=0
    def components(self):
        k=len(self.c); w=np.full(k+1,(1-STICKY)/k); w[self.last]=STICKY
        p=np.stack([norm((factor(c) if self.factorized else c)+U) for c in self.c]+[U])
        return w,p
    def prior(self):
        w,p=self.components(); return w@p
    def update(self,e):
        w,p=self.components(); j=int(np.argmax(w*(p@e))); credit=e/e.sum()
        if j==len(self.c): self.c.append(credit); self.births+=1
        else: self.c[j]+=credit
        self.switches+=int(j!=self.last); self.last=j

def test_e4():
    e=np.zeros(D); e[0]=1; f=np.zeros(D); f[1]=1; c=48*e
    a=Context(c); before=a.prior().copy(); a.update(f)
    assert a.births==1 and a.last==1 and a.prior()[1]>before[1]
    a.update(e); assert a.last==0 and len(a.c)==2
    b=ChangePoint(c); b.update(f); assert np.isclose(b.w.sum(),1) and b.w[0]>.5
    for obj in (a,b,Forget(c),Fixed(c)):
        p=obj.prior(); assert p.shape==(D,) and np.isfinite(p).all() and np.isclose(p.sum(),1) and (p>0).all()
    print('E4 normalization, contradiction, new context and old-context return: PASS')

BANK={b:macro_bank(acquire_source(PRIMITIVES[b])) for b in TRAIN_BASES+TEST_BASES}
PAIR_IDX={k:[M+TRANSFORMS.index(i)*M+TRANSFORMS.index(j) for i,j in v] for k,v in PAIR_FAMILIES.items()}
def tasks(seed,schedule):
    rng=np.random.default_rng(seed)
    for family in schedule:
        b=rng.choice(TEST_BASES); bank=BANK[b]
        i=int(rng.choice(PAIR_IDX[family])) if family!='random' else int(rng.integers(D))
        target=bank[i]; perm=rng.permutation(N); train,test=perm[:8],perm[8:]
        alive=consistent(bank,train[:1],target[train[:1]])
        full=consistent(bank,np.arange(N),target)
        acc=np.all(bank[:,test]==target[None,test],axis=2).mean(1)
        yield alive,full,acc,rng.random(D)

def evaluate(seed,n=48):
    c=learn_macros(PAIR_FAMILIES['matched'],seed=101,n=48)[0]-1
    results=[]
    schedules={'stable':['matched']*60,'ABA':['matched']*20+['swapped']*20+['matched']*20,'random':['random']*60}
    for regime,schedule in schedules.items():
        for rep in range(n):
            arms={'uniform':Fixed(np.zeros(D)),'fixed factor':Fixed(factor(c)), 'fixed joint':Fixed(c), 'forget':Forget(c), 'change point':ChangePoint(c), 'context factor':Context(c,True),'context joint':Context(c)}
            for t,(alive,full,acc,tie) in enumerate(tasks(seed+rep,schedule)):
                for name,model in arms.items():
                    p=model.prior(); eligible=np.flatnonzero(alive)
                    pred=eligible[np.lexsort((tie[eligible],p[eligible]))[-1]]
                    results.append((regime,rep,t,name,acc[pred],getattr(model,'births',0),getattr(model,'switches',0)))
                    # Completed-task evidence is available only AFTER scoring.
                    model.update(full)
    return pd.DataFrame(results,columns=['regime','rep','task','arm','exact','births','switches'])

def summarize(df):
    a=df.copy(); a['phase']=a.task//20
    table=a.groupby(['regime','phase','arm']).exact.mean().unstack('arm')*100
    print(table.round(1).to_string())
    curve=a[a.regime=='ABA'].groupby(['task','arm']).exact.mean().unstack('arm')
    for name in ['forget','change point','context factor','context joint']:
        recovery=[]
        for start in (20,40):
            v=curve[name].to_numpy(); recovery.append(next((j-start for j in range(start,start+18) if (v[j:j+3]>=.9).all()),None))
        print(name,'recovery (offset, 3 consecutive >=90%):',recovery)
    delta=a.pivot(index=['regime','rep','task'],columns='arm',values='exact')
    for regime in ['stable','ABA','random']:
        d=(delta.loc[regime]['context joint']-delta.loc[regime]['change point']).groupby('rep').mean().to_numpy()*100
        boot=np.random.default_rng(99).choice(d,(2000,len(d)),replace=True).mean(1)
        print(regime,'context minus change point pp:',round(d.mean(),2),np.round(np.quantile(boot,[.025,.975]),2))
    print('context births/switches mean at final task:')
    print(a[(a.task==59)&a.arm.str.startswith('context')].groupby(['regime','arm'])[['births','switches']].mean().round(2).to_string())
    return table


In [ ]:
from pathlib import Path
from threadpoolctl import threadpool_limits
test_e4()
# Seed 20261130 overlaps the first seed's streams; retain for audit, not replication.
with threadpool_limits(limits=1):
    for seed in (20261129,20261130,20261230):
        t=perf_counter(); print('\nSEED',seed)
        frame=evaluate(seed); summarize(frame)
        frame.to_csv(Path(f'e4_{seed}.csv'),index=False)
        print('seconds',round(perf_counter()-t,2))


## Measured decision — CPU executions, no GPU
All unit tests passed. Pilot and independent replication (48 streams each; seed ranges do not overlap). The intervening 20261130 range overlaps 47/48 pilot streams and is not independent.
| Seed | Arm | A | B | Return A | Shift / return recovery |
|---|---|---:|---:|---:|---|
| 20261129 | fixed joint |96.3|56.4|96.0| — |
| | forget |96.5|84.7|93.0|14 / 4|
| | change point |96.2|87.4|93.3|9 / 4|
| | context joint |96.2|85.6|95.3|13 / 2|
| 20261230 | fixed joint |95.7|58.8|96.6| — |
| | forget |95.4|85.9|93.7|10 / 4|
| | change point |95.6|88.9|93.5|10 / 4|
| | context joint |95.6|86.1|96.4|12 / 1|

Stable context accuracy averaged 96.0% and 96.2% across three phases. Random context minus change point: -2.19 pp (95% stream bootstrap [-3.07,-1.25]) and -2.30 pp [-3.32,-1.32]. A-B-A overall context minus change point: +0.08 pp [-0.49,+0.67] and +0.04 pp [-0.53,+0.57]. No overall demonstrated win. Context factorized recovery never reached criterion. Context joint created mean 1.10/1.08 contexts in A-B-A and 3.08/3.04 in random, suggesting spurious regime discovery. Context return gate passes twice; first-shift gate fails twice; stable gate meets target on phase-average (rounded); random control shows small harm. CPU ~19 seconds/seed, original pilot command timed out before producing output; reran with single-thread BLAS; algorithm unchanged. No claim of ARC, SOTA, or unrestricted concept discovery. Full truth-table feedback, fixed candidate bank, stationary meta-training and fixed hazard are limitations.

**Decision:** retain joint macro priors and the change-point baseline. Do not scale the hard context allocator. Next test soft context assignment or explicit evidence-calibrated reactivation against Bayesian change-point filtering on a new nonoverlapping seed range, with same feedback and context count penalty. Compare cumulative regret, first-shift and return separately. After that test genuine program synthesis rather than fixed-bank ranking.